<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/DayChallenge_Week6_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installation des dépendances

In [9]:
# Install necessary libraries
%pip install peft==0.4.0 transformers==4.27.0 datasets accelerate

# Create cache directory if it doesn't exist
!mkdir -p cache

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 6.2 MB/s eta 0:00:00
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 21.5 MB/s eta 0:00:00
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


## Importations et chargement du modèle/tokenizer

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import peft
from peft import LoraConfig, get_peft_model, PeftModel
import os
import datetime

# Define model name
model_name = "bigscience/bloomz-560m"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set pad_token to eos_token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load foundation model
foundation_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

ImportError: cannot import name 'PeftMixedModel' from 'peft' (/usr/local/lib/python3.12/dist-packages/peft/__init__.py)

## Chargement et prétraitement des données

In [3]:
# Load dataset
data = load_dataset("Abirate/english_quotes")

# Sample 10% of the training data
train_data = data["train"].shuffle(seed=42).select(range(int(len(data["train"]) * 0.1)))

# Tokenize the dataset
def tokenize_function(samples):
    return tokenizer(samples["quote"], truncation=True)

tokenized_train_data = train_data.map(tokenize_function, batched=True)

# Display a sample of the tokenized data
display(tokenized_train_data.select(range(5)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

## Configuration et application de LoRA

In [4]:
# Configure LoRA
lora_config = LoraConfig(
    r=8, # Rank of the update matrices. Common values are 8, 16, 32, 64.
    lora_alpha=16, # a scaling factor that adjusts the magnitude of the weight matrix. Usually set to r*2 or 16.
    target_modules=["query_key_value"], # Target specific layers for LoRA adaptation.
    lora_dropout=0.05,
    bias="none", # this specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM"
)

# Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

NameError: name 'LoraConfig' is not defined

## Configuration des arguments d'entraînement et entraînement du modèle

In [5]:
# Define output directory
output_directory = os.path.join("cache", "peft_lab_outputs")

# Configure Training Arguments
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True, # Automatically finds an appropriate batch size
    learning_rate=3e-2, # Higher learning rate than full fine-tuning.
    num_train_epochs=3, # Number of training epochs
    use_cpu=True # Use CPU for training
)

# Initialize Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train_data,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# Train the model
trainer.train()

NameError: name 'os' is not defined

## Sauvegarde du modèle LoRA optimisé

In [6]:
# Save the PEFT model
time_now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print(f"Model saved to: {peft_model_path}")

NameError: name 'datetime' is not defined

## Chargement du modèle LoRA et inférence

In [7]:
# Load the PEFT model for inference

# First, load the base model again to ensure it's clean for loading the adapter
# If you run this cell independently, uncomment the lines below
# foundation_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

# Load the LoRA adapter onto the base model
loaded_peft_model = PeftModel.from_pretrained(foundation_model, peft_model_path, is_trainable=False)

# Merge the adapter weights with the base model weights
merged_model = loaded_peft_model.merge_and_unload()

# Generate output tokens
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = merged_model.generate(
    **inputs,
    max_new_tokens=50, # Generate up to 50 new tokens
    num_beams=5, # Use beam search for better quality
    early_stopping=True, # Stop generation when all beam hypotheses have finished
    no_repeat_ngram_size=2, # Prevent repeating ngrams
    temperature=0.7, # Control randomness
    top_k=50, # Consider top_k tokens
    top_p=0.95 # Use nucleus sampling
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

NameError: name 'PeftModel' is not defined